# Week 1 companion notebook

Probability transport, flow maps, and the continuity equation.

This notebook reproduces every figure in the Week 1 notes and is meant to be edited. Change a velocity field, a step count, or a probe distribution and rerun the cell. Each section maps to a section of the notes.

Requires only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# site palette, so your plots match the notes
ACCENT, FG, MID, DIM = "#b3132a", "#18181b", "#3f3f46", "#6b6b71"
MUTED, LINE, STREAM = "#9a9aa0", "#e5e5e8", "#bcbcc2"
BLUE, BLUE_SOFT = "#2f6fb3", "#cfe0f0"
plt.rcParams.update({"font.size": 9, "figure.facecolor": "white",
                     "axes.labelcolor": MID, "xtick.color": DIM, "ytick.color": DIM})

def clean(ax, ticks=False):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(LINE)
    if not ticks: ax.set_xticks([]); ax.set_yticks([])


## 1. Simulating a flow: Euler, Heun, RK4

A flow is the solution of the ODE \(\dot X_t = u_t(X_t)\). For the linear field \(u(x)=Ax\) the exact flow is \(\psi_t(x_0)=e^{tA}x_0\), so we can measure the error of each integrator. Euler is first order, Heun second, RK4 fourth. The slopes on the log-log plot read off those orders directly. This is the engine behind every Week 1 coding lab.

In [ ]:
a, w = 0.35, 1.9                       # contraction + rotation; exact flow known
def vel(P):
    x, y = P[..., 0], P[..., 1]
    return np.stack([-a*x - w*y, w*x - a*y], -1)
def exact(t, x0):
    r = np.exp(-a*t)
    return r * np.array([np.cos(w*t)*x0[0] - np.sin(w*t)*x0[1],
                         np.sin(w*t)*x0[0] + np.cos(w*t)*x0[1]])

def integrate(method, N, x0, T):
    h = T/N; X = np.asarray(x0, float).copy(); pts=[X.copy()]
    for _ in range(N):
        if method == "euler":
            X = X + h*vel(X)
        elif method == "heun":
            k1 = vel(X); k2 = vel(X + h*k1); X = X + h/2*(k1 + k2)
        else:
            k1=vel(X); k2=vel(X+h/2*k1); k3=vel(X+h/2*k2); k4=vel(X+h*k3)
            X = X + h/6*(k1 + 2*k2 + 2*k3 + k4)
        pts.append(X.copy())
    return np.array(pts)

x0, T = np.array([2.0, 0.0]), 2.6
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ts = np.linspace(0, T, 400); ex = np.array([exact(t, x0) for t in ts])
ax1.plot(ex[:,0], ex[:,1], color=FG, lw=1.8, label="exact", zorder=5)
for m, c, lab in [("euler",ACCENT,"Euler"), ("heun",BLUE,"Heun"), ("rk4","#1f9d6b","RK4")]:
    p = integrate(m, 12, x0, T)
    ax1.plot(p[:,0], p[:,1], color=c, lw=1.1, marker="o", ms=3, label=f"{lab}, N=12")
ax1.set_aspect("equal"); ax1.legend(frameon=False, fontsize=8, loc="lower right")
ax1.set_title("same step count, different accuracy"); clean(ax1)

Ns = np.array([2,4,8,16,32,64,128,256]); xe = exact(T, x0)
for m, c, lab in [("euler",ACCENT,"Euler (1)"), ("heun",BLUE,"Heun (2)"), ("rk4","#1f9d6b","RK4 (4)")]:
    errs = [np.linalg.norm(integrate(m, int(N), x0, T)[-1] - xe) for N in Ns]
    ax2.loglog(Ns, errs, color=c, marker="o", ms=4, label=lab)
ax2.set_xlabel("steps N"); ax2.set_ylabel("error at t=T")
ax2.legend(frameon=False, fontsize=8); ax2.grid(True, which="both", color=LINE, lw=0.5, alpha=0.6)
clean(ax2, ticks=True); plt.tight_layout(); plt.show()


## 2. The instantaneous change of variables

Along a trajectory the log-density obeys \(\frac{d}{dt}\log p_t(X_t) = -\nabla\!\cdot u_t(X_t)\). For the contraction \(u(x)=-\theta x\) in two dimensions the divergence is \(-2\theta\), so the tracked log-density rises at rate \(2\theta\) and matches the analytic Gaussian exactly. Integrate the augmented state \((X_t, \log p_t)\) together.

In [ ]:
th = 0.9; rng = np.random.default_rng(1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
P0 = rng.standard_normal((1400, 2)); T = 1.8
for tt, al in [(0.0,0.16), (0.9,0.28), (T,0.5)]:
    Pt = np.exp(-th*tt)*P0
    ax1.scatter(Pt[:,0], Pt[:,1], s=4, color=ACCENT, alpha=al, edgecolors="none")
ax1.set_xlim(-3,3); ax1.set_ylim(-3,3); ax1.set_aspect("equal")
ax1.set_title("contraction concentrates the density"); clean(ax1)

ts = np.linspace(0, T, 200)
for x0 in P0[:6]:
    Xt = np.exp(-th*ts)[:,None]*x0; var = np.exp(-2*th*ts)
    logp_exact = -np.log(2*np.pi) - np.log(var) - 0.5*(Xt**2).sum(1)/var
    logp_track = logp_exact[0] + 2*th*ts                 # integrate -div u
    ax2.plot(ts, logp_exact, color=BLUE, lw=2.4, alpha=0.5)
    ax2.plot(ts, logp_track, color=ACCENT, lw=1.0, ls=(0,(4,2)))
ax2.plot([], [], color=BLUE, lw=2.4, alpha=0.5, label="analytic")
ax2.plot([], [], color=ACCENT, lw=1.0, ls=(0,(4,2)), label="tracked via -div u")
ax2.set_xlabel("time t"); ax2.set_ylabel("log p_t(X_t)")
ax2.legend(frameon=False, fontsize=8.5, loc="upper left"); clean(ax2, ticks=True)
plt.tight_layout(); plt.show()


## 3. The Hutchinson trace estimator

The CNF log-density needs the divergence \(\mathrm{tr}(\partial u/\partial x)\). The exact trace costs one autodiff pass per dimension. Hutchinson's identity \(\mathrm{tr}(A)=\mathbb{E}_\epsilon[\epsilon^\top A\,\epsilon]\) (for any \(\epsilon\) with \(\mathbb{E}[\epsilon\epsilon^\top]=I\)) turns that into one vector-Jacobian product. The estimate is unbiased but noisy, and Rademacher probes beat Gaussian ones because they cancel the diagonal's variance. First we verify the identity numerically, then plot convergence and variance.

In [ ]:
rng = np.random.default_rng(4); D = 24
A = rng.standard_normal((D, D)); np.fill_diagonal(A, rng.standard_normal(D)*5.0)
true = np.trace(A)

# verify tr(A) = E[eps^T A eps]
eps = rng.choice([-1.0, 1.0], (200000, D))
print("exact trace      ", round(true, 4))
print("Monte Carlo trace", round(np.einsum("ki,ij,kj->k", eps, A, eps).mean(), 4))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
M = 4000
for kind, c in [("Rademacher", ACCENT), ("Gaussian", BLUE)]:
    run, ests = 0.0, np.empty(M)
    for i in range(M):
        e = rng.choice([-1.0,1.0], D) if kind=="Rademacher" else rng.standard_normal(D)
        run += e @ A @ e; ests[i] = run/(i+1)
    ax1.plot(np.arange(1, M+1), ests, color=c, lw=1.0, label=kind)
ax1.axhline(true, color=FG, lw=1.1, ls=(0,(5,3))); ax1.set_xscale("log")
ax1.set_xlabel("probe vectors"); ax1.set_ylabel("running estimate of tr(A)")
ax1.legend(frameon=False, fontsize=8.5); ax1.grid(True, which="both", color=LINE, lw=0.5, alpha=0.5)
clean(ax1, ticks=True)

K = 20000
epsR = rng.choice([-1.0,1.0], (K, D)); epsG = rng.standard_normal((K, D))
vR = np.einsum("ki,ij,kj->k", epsR, A, epsR); vG = np.einsum("ki,ij,kj->k", epsG, A, epsG)
ax2.hist(vG, bins=60, color=BLUE_SOFT, edgecolor=BLUE, lw=0.5, density=True, label=f"Gaussian (var {vG.var():.0f})")
ax2.hist(vR, bins=60, color="#f6d4da", edgecolor=ACCENT, lw=0.5, density=True, alpha=0.8, label=f"Rademacher (var {vR.var():.0f})")
ax2.axvline(true, color=FG, lw=1.1, ls=(0,(5,3))); ax2.set_xlabel("single-probe value"); ax2.set_ylabel("density")
ax2.legend(frameon=False, fontsize=8.5); clean(ax2, ticks=True); plt.tight_layout(); plt.show()


## 4. Invariance and equivariance

A scientific density should be invariant, \(p(gx)=p(x)\), and the way to guarantee it is an equivariant velocity field, \(u(gx)=g\,u(x)\). Equivariance means the flow commutes with the symmetry. Below, the contraction is rotation equivariant so rotating then flowing equals flowing then rotating (blue dots land in the red circles). The shear is not, so the order matters.

In [ ]:
rng = np.random.default_rng(2)
pts = []
while len(pts) < 6:
    p = rng.uniform(-1.3, 1.3, 2)
    if all(np.linalg.norm(p-q) > 0.7 for q in pts): pts.append(p)
X = np.array(pts)
ang = np.deg2rad(80); Q = np.array([[np.cos(ang),-np.sin(ang)],[np.sin(ang),np.cos(ang)]])

def flow(P, A, T=1.0, steps=60):
    h = T/steps; Z = np.asarray(P, float).copy()
    for _ in range(steps): Z = Z + h*(Z @ A.T)
    return Z
A_eq  = np.array([[-0.35, 0.0],[0.0, -0.35]])     # contraction: rotation equivariant
A_neq = np.array([[0.0, 0.9],[0.0, 0.0]])         # shear: not

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, A, title in [(ax1, A_eq, "equivariant: u(Qx) = Q u(x)"), (ax2, A_neq, "non-equivariant: u(Qx) != Q u(x)")]:
    QphiX = flow(X, A) @ Q.T           # flow then rotate
    phiQX = flow(X @ Q.T, A)           # rotate then flow
    ax.scatter(X[:,0], X[:,1], s=34, color=MUTED, label="x")
    ax.scatter((X@Q.T)[:,0], (X@Q.T)[:,1], s=34, color="#9bbad8", label="Qx")
    ax.scatter(QphiX[:,0], QphiX[:,1], s=130, facecolors="none", edgecolors=ACCENT, lw=1.4, label="Q phi(x)")
    ax.scatter(phiQX[:,0], phiQX[:,1], s=30, color=BLUE, label="phi(Qx)")
    ax.set_aspect("equal"); ax.set_xlim(-2.2,2.2); ax.set_ylim(-2.2,2.2)
    ax.set_title(title); ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2); clean(ax)
plt.tight_layout(); plt.show()


## Play with it

A few things to try.

- Swap `vel` in section 1 for a nonlinear field and integrate the augmented state to confirm the log-density identity numerically.
- In section 3, shrink the matrix dimension `D` and watch the estimator variance fall. Replace `A` with the Jacobian of a small neural network and form `eps @ A` with an autodiff vector-Jacobian product.
- In section 4, build an equivariant field from pairwise differences, \(u_i = \sum_{j} \phi(\lVert r_i-r_j\rVert)(r_j - r_i)\), and check that it also commutes with rotation.